### 635. Design Log Storage System

<p>You are given several logs, where each log contains a unique ID and timestamp. Timestamp is a string that has the following format: <code>Year:Month:Day:Hour:Minute:Second</code>, for example, <code>2017:01:01:23:59:59</code>. All domains are zero-padded decimal numbers.</p>

<p>Implement the <code>LogSystem</code> class:</p>

<ul>
	<li><code>LogSystem()</code> Initializes the <code>LogSystem</code><b> </b>object.</li>
	<li><code>void put(int id, string timestamp)</code> Stores the given log <code>(id, timestamp)</code> in your storage system.</li>
	<li><code>int[] retrieve(string start, string end, string granularity)</code> Returns the IDs of the logs whose timestamps are within the range from <code>start</code> to <code>end</code> inclusive. <code>start</code> and <code>end</code> all have the same format as <code>timestamp</code>, and <code>granularity</code> means how precise the range should be (i.e. to the exact <code>Day</code>, <code>Minute</code>, etc.). For example, <code>start = &quot;2017:01:01:23:59:59&quot;</code>, <code>end = &quot;2017:01:02:23:59:59&quot;</code>, and <code>granularity = &quot;Day&quot;</code> means that we need to find the logs within the inclusive range from <strong>Jan. 1st 2017</strong> to <strong>Jan. 2nd 2017</strong>, and the <code>Hour</code>, <code>Minute</code>, and <code>Second</code> for each log entry can be ignored.</li>
</ul>

<p>&nbsp;</p>
<p><strong class="example">Example 1:</strong></p>

<pre>
<strong>Input</strong>
[&quot;LogSystem&quot;, &quot;put&quot;, &quot;put&quot;, &quot;put&quot;, &quot;retrieve&quot;, &quot;retrieve&quot;]
[[], [1, &quot;2017:01:01:23:59:59&quot;], [2, &quot;2017:01:01:22:59:59&quot;], [3, &quot;2016:01:01:00:00:00&quot;], [&quot;2016:01:01:01:01:01&quot;, &quot;2017:01:01:23:00:00&quot;, &quot;Year&quot;], [&quot;2016:01:01:01:01:01&quot;, &quot;2017:01:01:23:00:00&quot;, &quot;Hour&quot;]]
<strong>Output</strong>
[null, null, null, null, [3, 2, 1], [2, 1]]

<strong>Explanation</strong>
LogSystem logSystem = new LogSystem();
logSystem.put(1, &quot;2017:01:01:23:59:59&quot;);
logSystem.put(2, &quot;2017:01:01:22:59:59&quot;);
logSystem.put(3, &quot;2016:01:01:00:00:00&quot;);

// return [3,2,1], because you need to return all logs between 2016 and 2017.
logSystem.retrieve(&quot;2016:01:01:01:01:01&quot;, &quot;2017:01:01:23:00:00&quot;, &quot;Year&quot;);

// return [2,1], because you need to return all logs between Jan. 1, 2016 01:XX:XX and Jan. 1, 2017 23:XX:XX.
// Log 3 is not returned because Jan. 1, 2016 00:00:00 comes before the start of the range.
logSystem.retrieve(&quot;2016:01:01:01:01:01&quot;, &quot;2017:01:01:23:00:00&quot;, &quot;Hour&quot;);
</pre>

<p>&nbsp;</p>
<p><strong>Constraints:</strong></p>

<ul>
	<li><code>1 &lt;= id &lt;= 500</code></li>
	<li><code>2000 &lt;= Year &lt;= 2017</code></li>
	<li><code>1 &lt;= Month &lt;= 12</code></li>
	<li><code>1 &lt;= Day &lt;= 31</code></li>
	<li><code>0 &lt;= Hour &lt;= 23</code></li>
	<li><code>0 &lt;= Minute, Second &lt;= 59</code></li>
	<li><code>granularity</code> is one of the values <code>[&quot;Year&quot;, &quot;Month&quot;, &quot;Day&quot;, &quot;Hour&quot;, &quot;Minute&quot;, &quot;Second&quot;]</code>.</li>
	<li>At most <code>500</code> calls will be made to <code>put</code> and <code>retrieve</code>.</li>
</ul>


#### Difficulty: Medium, AC rate: 59.8%

#### Topics:
Hash Table | String | Design | Ordered Set

#### Links:
 🎁 [Question Detail](https://leetcode.com/problems/design-log-storage-system/description/) | 🎉 [Question Solution](https://leetcode.com/problems/design-log-storage-system/solution/) | 💬 [Question Discussion](https://leetcode.com/problems/design-log-storage-system/discuss/?orderBy=most_votes)



#### Test Case(s):

1. `["LogSystem","put","put","put","retrieve","retrieve"]`  
`[[],[1,"2017:01:01:23:59:59"],[2,"2017:01:01:22:59:59"],[3,"2016:01:01:00:00:00"],["2016:01:01:01:01:01","2017:01:01:23:00:00","Year"],["2016:01:01:01:01:01","2017:01:01:23:00:00","Hour"]]`

---
What's your idea?

Ordered by key mapping, like `TreeMap` from Java

---

In [16]:
from collections import defaultdict
import bisect
from datetime import datetime
import calendar

class LogSystem:
    FORMAT = "%Y:%m:%d:%H:%M:%S"
    GRANULARITIES = ["Year", "Month", "Day", "Hour", "Minute", "Second"]
    START_F = [
        lambda obj: obj.replace(month=1, day=1, hour=0, minute=0, second=0, microsecond=0), # start of year
        lambda obj: obj.replace(day=1, hour=0, minute=0, second=0, microsecond=0), # start of month
        lambda obj: obj.replace(hour=1, minute=0, second=0, microsecond=0), # start of day
        lambda obj: obj.replace(minute=0, second=0, microsecond=0), # start of hour
        lambda obj: obj.replace(second=0, microsecond=0), # start of minute
        lambda obj: obj.replace(microsecond=0), # start of second
    ]
    END_F = [
        lambda obj: datetime(obj.year, 12, 31, 23, 59, 59, 999999), # end of year
        lambda obj: datetime(obj.year, obj.month, calendar.monthrange(obj.year, obj.month)[1], 23, 59, 59, 999999), # end of month
        lambda obj: obj.replace(hour=23, minute=59, second=59, microsecond=999999), # end of day
        lambda obj: obj.replace(minute=59, second=59, microsecond=999999), # end of hour
        lambda obj: obj.replace(second=59, microsecond=999999), # end of minute
        lambda obj: obj.replace(microsecond=999999), # end of second
    ]
    
    def __init__(self):
        self.store = defaultdict(set)
        self.keys = []

    def put(self, id: int, timestamp: str) -> None:
        ts = datetime.strptime(timestamp, self.FORMAT).timestamp()
        if ts not in self.store:
            bisect.insort(self.keys, ts)
        self.store[ts].add(id)

    def retrieve(self, start: str, end: str, granularity: str) -> list[int]:
        start_obj, end_obj = datetime.strptime(start, self.FORMAT), datetime.strptime(end, self.FORMAT)
        g_idx = self.GRANULARITIES.index(granularity)
        start_ts, end_ts = self.START_F[g_idx](start_obj).timestamp(), self.END_F[g_idx](end_obj).timestamp()
        start_idx, end_idx = bisect.bisect_left(self.keys, start_ts), bisect.bisect_left(self.keys, end_ts)
        result = []
        for key in self.keys[start_idx:end_idx]:
            result.extend(self.store[key])
        return result

In [17]:
# Your LogSystem object will be instantiated and called as such:
# obj = LogSystem()
# obj.put(id,timestamp)
# param_2 = obj.retrieve(start,end,granularity)
obj = LogSystem()
for func, params in zip(["put","put","put","retrieve","retrieve"], [[1,"2017:01:01:23:59:59"],[2,"2017:01:01:22:59:59"],[3,"2016:01:01:00:00:00"],["2016:01:01:01:01:01","2017:01:01:23:00:00","Year"],["2016:01:01:01:01:01","2017:01:01:23:00:00","Hour"]]):
    print(f'{func}({params}): {getattr(obj, func)(*params)}')
    # [null, null, null, null, [3, 2, 1], [2, 1]]

put([1, '2017:01:01:23:59:59']): None
put([2, '2017:01:01:22:59:59']): None
put([3, '2016:01:01:00:00:00']): None
retrieve(['2016:01:01:01:01:01', '2017:01:01:23:00:00', 'Year']): [3, 2, 1]
retrieve(['2016:01:01:01:01:01', '2017:01:01:23:00:00', 'Hour']): [2, 1]


In [18]:
obj = LogSystem()
for func, params in zip(["put","put","retrieve"], [[1,"2017:01:01:23:59:59"],[2,"2017:01:02:23:59:59"],["2017:01:01:23:59:59","2017:01:02:23:59:59","Day"]]):
    print(f'{func}({params}): {getattr(obj, func)(*params)}')
    # [null,null,null,[1,2]]

put([1, '2017:01:01:23:59:59']): None
put([2, '2017:01:02:23:59:59']): None
retrieve(['2017:01:01:23:59:59', '2017:01:02:23:59:59', 'Day']): [1, 2]


😃 Result: Accepted  
💯 Passed Test Case: 65 / 65  
🚀 Runtime: 29 ms, Memory: 20.1 MB  
🉑 Runtime Percentile: better than 5.28%, Memory Percentile: better than 6.74%  
📆 Finished At: Wed, 07 Oct 2026 19:10:38 GMT